# Exercise 1 — Saturated P control of indoor temperature

First-order (1R1C) building model, identified from measured data:

$$T_{k+1} = A\,T_k + B\,u_k + E\,T_{o,k}, \qquad \Delta t = 1800\ \text{s}$$

- $T_k$: indoor temperature [°C] (state)
- $u_k$: heating power [W] (control input)
- $T_{o,k}$: outdoor temperature [°C] (disturbance)

**Goal:** keep the indoor temperature at $r = 21$ °C with a P controller limited by the heater power:

$$u_k = \mathrm{clip}\big(K_p\,(r_k - T_k),\; 0,\; u_{\max}\big), \qquad u_{\max} = 2000\ \text{W}$$

The outdoor temperature is the measured one shifted by −12 °C (a winter case, ≈ 1–21 °C).

In [ ]:
# Google Colab only: download the course files and check the packages.
# When you run the notebook on your own computer, this cell does nothing.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("building_model.py"):
        if not os.path.exists("athens_neural_network_for_control"):
            subprocess.run(["git", "clone", "-q", "https://github.com/li-yun/athens_neural_network_for_control.git"], check=True)
        os.chdir("athens_neural_network_for_control")
    r = subprocess.run([sys.executable, "install_packages.py"], capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out[out.find("2/3"):] if r.returncode == 0 else out)   # show the check results
    if r.returncode != 0:
        raise RuntimeError("Setup failed, see the messages above.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from building_model import FirstOrderBuilding, outdoor_temperature, plot_closed_loop, evaluate

model = FirstOrderBuilding()
print(model)

To = outdoor_temperature('winter')     # disturbance, dt = 30 min
r, u_max, T0 = 21.0, 2000.0, 15.0

## Task: implement the saturated P controller

*Hint:* `np.clip(x, lower, upper)`

In [ ]:
def p_controller(r, T, Kp, u_max):
    """Saturated P controller: u = clip(Kp*(r - T), 0, u_max)."""
    # TODO: one line
    return ...

In [ ]:
# Check
assert p_controller(21, 20, Kp=500, u_max=2000) == 500
assert p_controller(21, 10, Kp=500, u_max=2000) == 2000   # upper limit
assert p_controller(21, 23, Kp=500, u_max=2000) == 0      # heater cannot cool
print('p_controller OK')

## Closed-loop simulation

Run the cells below. Then try other values of $K_p$ and observe the steady-state error, the saturation and the oscillations at high gain.

In [ ]:
Kp = 1000.0   # W/K

res = model.simulate_closed_loop(T0, r, To, controller=lambda r_k, T_k: p_controller(r_k, T_k, Kp, u_max))
plot_closed_loop(res, labels=[f'Kp = {Kp:.0f} W/K'], u_max=u_max)
plt.show()

In [ ]:
gains = [200, 1000, 5000, 20000]
results = [model.simulate_closed_loop(T0, r, To, lambda r_k, T_k, Kp=Kp: p_controller(r_k, T_k, Kp, u_max))
           for Kp in gains]
plot_closed_loop(results, labels=[f'Kp = {Kp}' for Kp in gains], u_max=u_max)
plt.show()

# performance over the last 24 h (after the warm-up)
pd.DataFrame([evaluate(res, skip_hours=res['t'][-1] - 24) for res in results],
             index=[f'Kp = {Kp}' for Kp in gains]).round(3)